In [1]:
from PCA_utils import *

# Parameters

In [2]:
CONCATENATE_DATA = False
SAVE_CONCATENATED_DATA = False
CLEAN_DATA = False

features = [
    "n_foci",
    "area",
    "equivalent_diameter_area",
    "eccentricity",
    "perimeter",
    "solidity",
    "mean_intensity",
    "max_intensity",
    "min_intensity",
]

# Import data

In [ ]:
# Root directory where the data is stored
root_dir = "/mnt/c/Users/helen/Desktop/data"

# Path to the aggregated data files for each condition
data = [
        {'name': 'WT', 'path': f"{root_dir}/WT/gmm4/aggregated_data.csv"},
        {'name': 'WT, NC', 'path': f"{root_dir}/WT_NC/gmm4/aggregated_data.csv"},
        {'name': 'WT, Apt.3', 'path': f"{root_dir}/WT_Apt3/gmm4/aggregated_data.csv"},
        {'name': 'MGS3', 'path': f"{root_dir}/MGS3/gmm4/aggregated_data.csv"},
        {'name': 'MGS3, NC', 'path': f"{root_dir}/MGS3_NC/gmm4/aggregated_data.csv"},
        {'name': 'MGS3, Apt.3', 'path': f"{root_dir}/MGS3_Apt3/gmm4/aggregated_data.csv"},
    ]

# Add the labels for each condition to the data list
labels = [df['name'] for df in data]
print(f"Sample labels: {labels}")

# Load the dataframes from the specified paths
dfs = []
for item in data:
        p = item["path"]
        df = pd.read_csv(p)
        dfs.append(df)

# Print the number of dataframes loaded
N_files = len(dfs)
print(f"Loaded {N_files} dataframes.")

if CONCATENATE_DATA:
    df = pd.concat(
        [df.assign(sample=sample) for df, sample in zip(dfs, labels)],
        ignore_index=True
    )

    print(f"Concatenated dataframe shape: {df.shape}")

if CLEAN_DATA:
    #df = df.iloc[:, 2:]
    df.columns = df.columns.str.strip().str.lower()
    df.iloc[:, 1:9] = df.iloc[:, 1:9].apply(pd.to_numeric, errors="coerce")
    df = df.dropna()
    print(f"Cleaned dataframe shape: {df.shape}")
    
if SAVE_CONCATENATED_DATA:
    concatenated_path = f"{root_dir}/concatenated_data.csv"
    df.to_csv(concatenated_path, index=False)
    print(f"Concatenated dataframe saved to: {concatenated_path}")

In [3]:
# Root directory where the data is stored
root_dir = "/mnt/c/Users/helen/Desktop/data"

df = pd.read_csv(f"{root_dir}/concatenated_data.csv")

In [4]:
df.head()

,image_id,n_foci,area,equivalent_diameter_area,eccentricity,solidity,perimeter,mean_intensity,max_intensity,min_intensity,sample
0,c1_wt-01-airyscan_processing-29_roi_01_0836-12...,54,50.0,7.978846,0.850810,0.860612,29.142136,309.998943,394.5,247.5,WT
1,c1_wt-02-airyscan_processing-30_roi_01_0650-08...,73,59.0,8.667245,0.845191,0.847458,31.142136,322.148148,404.0,256.0,WT
2,c1_wt-02-airyscan_processing-30_roi_02_1538-15...,126,50.5,8.018542,0.834528,0.821144,27.435029,177.704891,233.0,137.5,WT
3,c1_wt-03-airyscan_processing-31_roi_01_1581-04...,17,153.0,13.957279,0.836873,0.838710,44.384776,3157.554960,4072.0,2379.0,WT
4,c1_wt-03-airyscan_processing-31_roi_02_0430-09...,64,77.0,9.901278,0.856178,0.815824,35.867009,356.250000,456.5,281.0,WT


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Prepare data
pca_df = df.dropna(
    subset=features + ["experiment", "sample"]
).copy()

pca_df["experiment"] = pca_df["experiment"].astype(str)

# Standardize ALL experiments together
X = pca_df[features]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Fit ONE PCA using all cells
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

pca_df["PC1"] = X_pca[:, 0]
pca_df["PC2"] = X_pca[:, 1]

# Explained variance
explained = pca.explained_variance_ratio_ * 100

# Plot PCA by experiment
plt.figure(figsize=(9, 7))

sns.scatterplot(
    data=pca_df,
    x="PC1",
    y="PC2",
    hue="experiment",
    palette="Set2",
    s=60,
    alpha=0.7
)

plt.axhline(0, color="gray", linestyle="--", alpha=0.4)
plt.axvline(0, color="gray", linestyle="--", alpha=0.4)

plt.xlabel(f"PC1 ({explained[0]:.1f}%)")
plt.ylabel(f"PC2 ({explained[1]:.1f}%)")
plt.title("Combined PCA — Colored by Experiment")

plt.legend(title="Experiment", bbox_to_anchor=(1.05, 1))
plt.tight_layout()
plt.show()

# Plot the SAME PCA by sample
plt.figure(figsize=(9, 7))

sns.scatterplot(
    data=pca_df,
    x="PC1",
    y="PC2",
    hue="sample",
    palette="Set2",
    s=60,
    alpha=0.7
)

plt.axhline(0, color="gray", linestyle="--", alpha=0.4)
plt.axvline(0, color="gray", linestyle="--", alpha=0.4)

plt.xlabel(f"PC1 ({explained[0]:.1f}%)")
plt.ylabel(f"PC2 ({explained[1]:.1f}%)")
plt.title("Combined PCA — Colored by Sample")

plt.legend(title="Sample", bbox_to_anchor=(1.05, 1))
plt.tight_layout()
plt.show()

In [ ]:
loadings = pd.DataFrame(
    pca.components_.T,
    index=features,
    columns=["PC1", "PC2"]
)

print(loadings.round(3))

In [ ]:
g = sns.relplot(
    data=pca_df,
    x="PC1",
    y="PC2",
    hue="sample",
    col="experiment",
    col_wrap=3,
    palette="Set2",
    kind="scatter",
    alpha=0.7,
    s=55,
    height=4,
    facet_kws={"sharex": True, "sharey": True}
)

g.set_axis_labels(
    f"PC1 ({explained[0]:.1f}%)",
    f"PC2 ({explained[1]:.1f}%)"
)

g.set_titles("Experiment {col_name}")

plt.show()

# Correlation analysis

In [ ]:
# Select numerical columns
df_numeric = df.select_dtypes(include=np.number)

# Calculate Spearman correlation
corr = df_numeric.corr(method="spearman")

display(corr.round(3))

In [ ]:
plt.figure(figsize=(12, 9))

# Show only lower triangle
mask = np.triu(np.ones_like(corr, dtype=bool))

sns.heatmap(
    corr,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    linewidths=0.5
)

plt.title("Spearman Correlation Matrix", fontsize=15)
plt.tight_layout()
plt.show()

## Graphs

In [ ]:
features = [
    "n_foci",
    "area",
    "equivalent_diameter_area",
    "eccentricity",
    "perimeter",
    "solidity",
    "mean_intensity",
    "max_intensity",
]

fig, axes = plt.subplots(2, 4, figsize=(10, 9))

for ax, feature in zip(axes.flat, features):

    sns.violinplot(
        data=df,
        y=feature,
        color="lightblue",
        inner="box",
        legend=False,
        ax=ax
    )
    
    ax.set_title(feature.replace("_", " ").title())
    ax.set_xlabel("")
    ax.set_ylabel(feature)

plt.tight_layout()
plt.show()


# PCA

In [ ]:
# Features
features = [
    "n_foci",
    #"area",
    "equivalent_diameter_area",
    "eccentricity",
    "perimeter",
    "solidity",
    "mean_intensity",
    #"max_intensity",
    #"min_intensity",
]

# Select features and scale them
pca_df = df.copy()

X = df[features]
X_scaled = StandardScaler().fit_transform(X)

# PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

pca_df["PC1"] = X_pca[:, 0]
pca_df["PC2"] = X_pca[:, 1]

In [ ]:
# PCA coefficients
loadings = pd.DataFrame(
    pca.components_.T,
    index=features,
    columns=['PC1', 'PC2']
)

#display(loadings.round(3))

# Heatmap of PCA loadings
plt.figure(figsize=(8, 7))

sns.heatmap(
    loadings[["PC1", "PC2"]],
    annot=True,
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    fmt=".2f"
)

plt.title("PCA Feature Loadings")
plt.tight_layout()
plt.show()

In [ ]:
centroids = pca_df.groupby("sample")[["PC1", "PC2"]].mean()

print(centroids)

In [ ]:
#centroids = plot_pca_centroids_ci(
    #pca_df,
    #group_col="sample",
    #confidence=0.95
#)

In [ ]:
results = bootstrap_pca_centroids(
    pca_df,
    group_col="sample",
    pca=pca,
    confidence=0.95,
    palette="Set2",
    centroid_size=65,
    show_bootstrap=False
)

In [ ]:
results_stats = pca_permanova(
    pca_df,
    group_col="sample"
)

results_stats

In [ ]:
subset = pca_df[(pca_df['sample'] == "WT")| (pca_df['sample'] == "MGS3")]

results_stats = pca_permanova(
    subset,
    group_col="experiment"
)

results_stats

### Centroids distances tree

In [ ]:
distances = plot_centroid_distance_tree(
    results,
    reference="WT",
    palette="Set2"
)

# Detect outliers

In [ ]:
pca_df_outliers = detect_pca_outliers(
    pca_df,
    group_col="sample",
    features=("PC1", "PC2"),
    confidence=0.99
)

# Display detected outliers
outliers = pca_df_outliers[
    pca_df_outliers["outlier"]
]

display(outliers)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

sns.scatterplot(
    data=pca_df_outliers,
    x="PC1",
    y="PC2",
    hue="sample",
    alpha=0.5,
    s=45,
    ax=ax
)

# Highlight outliers
outliers = pca_df_outliers[pca_df_outliers["outlier"]]

ax.scatter(
    outliers["PC1"],
    outliers["PC2"],
    s=130,
    facecolors="none",
    edgecolors="red",
    linewidths=2,
    label="Outliers"
)

ax.axhline(0, color="gray", linestyle="--", alpha=0.4)
ax.axvline(0, color="gray", linestyle="--", alpha=0.4)

ax.set_title("PCA — Multivariate Outlier Detection")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
df_clean = df.loc[~df.index.isin(outliers.index)].copy()
df_clean.shape

In [ ]:
df_clean = df_clean[~(df_clean['sample'] == 'MGS1')]

In [ ]:
# Features
features = [
    "n_foci",
    #"area",
    "equivalent_diameter_area",
    "eccentricity",
    "perimeter",
    "solidity",
    "mean_intensity",
    #"max_intensity",
]

# Select features and scale them
pca_df = df_clean.copy()

X = df_clean[features]
X_scaled = StandardScaler().fit_transform(X)

# PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

pca_df["PC1"] = X_pca[:, 0]
pca_df["PC2"] = X_pca[:, 1]

In [ ]:
# PCA coefficients
loadings = pd.DataFrame(
    pca.components_.T,
    index=features,
    columns=['PC1', 'PC2']
)

#display(loadings.round(3))

# Heatmap of PCA loadings
plt.figure(figsize=(8, 7))

sns.heatmap(
    loadings[["PC1", "PC2"]],
    annot=True,
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    fmt=".2f"
)

plt.title("PCA Feature Loadings")
plt.tight_layout()
plt.show()

In [ ]:
results = bootstrap_pca_centroids(
    pca_df,
    group_col="sample",
    n_boot=5000,
    confidence=0.95,
    pca = pca
)

In [ ]:
distances = plot_centroid_distance_tree(
    results,
    reference="WT",
    palette="Set2"
)

In [ ]:
centroids = pca_df.groupby("sample")[["PC1", "PC2"]].mean()

print(centroids)

In [ ]:
results_stats = pca_permanova(
    pca_df,
    group_col="sample"
)

results_stats

In [ ]:
subset = pca_df[(pca_df['sample'] == "WT") | (pca_df['sample'] == "MGS2")]

results_stats = pca_permanova(
    subset,
    group_col="sample"
)

results_stats

In [ ]:
subset = pca_df[(pca_df['sample'] == "WT") | (pca_df['sample'] == "MGS5")]

results_stats = pca_permanova(
    subset,
    group_col="sample"
)

results_stats

In [ ]:
subset = pca_df[(pca_df['sample'] == "WT") | (pca_df['sample'] == "MGS3")]

results_stats = pca_permanova(
    subset,
    group_col="sample"
)

results_stats

In [ ]:
subset = pca_df[(pca_df['sample'] == "WT") | (pca_df['sample'] == "MGS2")]

results_stats = pca_permanova(
    subset,
    group_col="sample"
)

results_stats

In [ ]:
subset = pca_df[(pca_df['sample'] == "WT") | (pca_df['sample'] == "MGS1")]

results_stats = pca_permanova(
    subset,
    group_col="sample"
)

results_stats